In [2]:
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

df = pd.read_csv("/content/drive/MyDrive/CEAS_08.csv")

print(df.shape)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(39154, 7)


,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd

train_df = pd.read_csv("/content/drive/MyDrive/spam_train_cleaned.csv")
test_df = pd.read_csv("/content/drive/MyDrive/spam_test_cleaned.csv")

print("Train:", train_df.shape)
print("Test:", test_df.shape)

Train: (31311, 16)
Test: (7828, 16)


In [5]:
print("Train duplicates:", train_df["combined_text"].duplicated().sum())
print("Test duplicates:", test_df["combined_text"].duplicated().sum())

Train duplicates: 3900
Test duplicates: 793


In [6]:
/# Remove duplicate emails from train and test
train_df = train_df.drop_duplicates(subset="combined_text").reset_index(drop=True)
test_df = test_df.drop_duplicates(subset="combined_text").reset_index(drop=True)

print("Train shape after removing duplicates:", train_df.shape)
print("Test shape after removing duplicates:", test_df.shape)

print("Train duplicates:", train_df["combined_text"].duplicated().sum())
print("Test duplicates:", test_df["combined_text"].duplicated().sum());

Train shape after removing duplicates: (27411, 16)
Test shape after removing duplicates: (7035, 16)
Train duplicates: 0
Test duplicates: 0


In [7]:
# check overlap

# Check duplicates within each dataset
print("Train duplicates:", train_df["combined_text"].duplicated().sum())
print("Test duplicates :", test_df["combined_text"].duplicated().sum())

# Check overlap between train and test
train_texts = set(train_df["combined_text"])
test_texts = set(test_df["combined_text"])

overlap = train_texts.intersection(test_texts)

print("Train-Test duplicated texts:", len(overlap))

#Delete overlapp
df = pd.concat([train_df, test_df], ignore_index=True)

# Remove exact duplicate emails
df = df.drop_duplicates(subset="combined_text")

# Create a fresh clean split
from sklearn.model_selection import train_test_split

train_clean, test_clean = train_test_split(
    df,
    test_size=0.20,
    stratify=df["label"],
    random_state=42
)

Train duplicates: 0
Test duplicates : 0
Train-Test duplicated texts: 388


In [8]:
print("Train class distribution:")
print(train_df["label"].value_counts())

print("\nTest class distribution:")
print(test_df["label"].value_counts())

Train class distribution:
label
0    13790
1    13621
Name: count, dtype: int64

Test class distribution:
label
1    3580
0    3455
Name: count, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix


# X and y


X = train_df.drop(columns=["label"])
y = train_df["label"]

X_test_raw = test_df.drop(columns=["label"])
y_test = test_df["label"]



# Train / Validation split


X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



# Text + numerical columns


TEXT_COL = "combined_text"

NUMERIC_COLS = [
    col for col in X.columns
    if col != TEXT_COL
]



# TF-IDF


tfidf = TfidfVectorizer(
    max_features=300,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_text_train = tfidf.fit_transform(X_train_raw[TEXT_COL])
X_text_val = tfidf.transform(X_val_raw[TEXT_COL])
X_text_test = tfidf.transform(X_test_raw[TEXT_COL])


# scale
scaler = StandardScaler()

X_num_train = scaler.fit_transform(X_train_raw[NUMERIC_COLS])
X_num_val = scaler.transform(X_val_raw[NUMERIC_COLS])
X_num_test = scaler.transform(X_test_raw[NUMERIC_COLS])


# Combine ALL features


X_train = hstack([
    X_text_train,
    csr_matrix(X_num_train)
])

X_val = hstack([
    X_text_val,
    csr_matrix(X_num_val)
])

X_test = hstack([
    X_text_test,
    csr_matrix(X_num_test)
])


print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Training: (21928, 314)
Validation: (5483, 314)
Test: (7035, 314)


In [10]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [11]:
from sklearn.metrics import accuracy_score, recall_score, f1_score

train_pred = rf.predict(X_train)
test_pred = rf.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy:", accuracy_score(y_test, test_pred))

print("Train Recall:", recall_score(y_train, train_pred))
print("Test Recall:", recall_score(y_test, test_pred))

print("Train F1:", f1_score(y_train, train_pred))
print("Test F1:", f1_score(y_test, test_pred))

Train Accuracy: 1.0
Test Accuracy: 0.9901918976545843
Train Recall: 1.0
Test Recall: 0.9930167597765364
Train F1: 1.0
Test F1: 0.9903886335144171


hyperprameters

In [12]:
rf_depth = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42
)

rf_depth.fit(X_train, y_train)

train_pred = rf_depth.predict(X_train)
test_pred = rf_depth.predict(X_test)

print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy:", accuracy_score(y_test, test_pred))

print("Train Recall:", recall_score(y_train, train_pred))
print("Test Recall:", recall_score(y_test, test_pred))

print("Train F1:", f1_score(y_train, train_pred))
print("Test F1:", f1_score(y_test, test_pred))

Train Accuracy: 0.9961692812842029
Test Accuracy: 0.9890547263681592
Train Recall: 0.9998164464023495
Test Recall: 0.9932960893854749
Train F1: 0.9961594732991953
Test F1: 0.9892891918208374


cross validation

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

param_grid = {
    "n_estimators": [200, 500],
    "max_depth": [None, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt"],
    "class_weight": [None],
}
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_rf = GridSearchCV(
    rf,
    param_grid,
    scoring={
        "f1": "f1",
        "recall": "recall",
        "accuracy": "accuracy"
    },
    refit="f1",
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
    verbose=1
)

grid_rf.fit(X_train, y_train)

print("Best params:", grid_rf.best_params_)
print("Best CV F1: %.4f" % grid_rf.best_score_)

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best params: {'class_weight': None, 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best CV F1: 0.9889


In [15]:
from sklearn.metrics import classification_report

# Full CV results, best first
cv_results_rf = pd.DataFrame(grid_rf.cv_results_)

cols_rf = [
    "param_n_estimators",
    "param_max_depth",
    "param_min_samples_split",
    "param_min_samples_leaf",
    "param_max_features",
    "param_class_weight",
    "mean_train_f1",
    "mean_test_f1",
    "std_test_f1",
    "mean_test_recall",
    "mean_test_accuracy",
]

print(
    cv_results_rf[cols_rf]
    .sort_values("mean_test_f1", ascending=False)
    .head(10)
    .to_string(index=False)
)

print(
    classification_report(
        y_test,
        grid_rf.predict(X_test),
        target_names=["ham", "spam"]
    )
)

 param_n_estimators param_max_depth  param_min_samples_split  param_min_samples_leaf param_max_features param_class_weight  mean_train_f1  mean_test_f1  std_test_f1  mean_test_recall  mean_test_accuracy
                200            None                        2                       1               sqrt               None       1.000000      0.988942     0.002758          0.992566            0.988964
                500            None                        2                       1               sqrt               None       1.000000      0.988898     0.002389          0.992658            0.988918
                500            None                        5                       1               sqrt               None       0.999794      0.988300     0.002354          0.991832            0.988325
                200            None                        5                       1               sqrt               None       0.999736      0.988206     0.002617          0.991373      

In [16]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, classification_report

best_rf = grid_rf.best_estimator_

for name, X_, y_ in [
    ("Train", X_train, y_train),
    ("Val  ", X_val, y_val),
    ("Test ", X_test, y_test),
]:
    pred = best_rf.predict(X_)
    print("%s  acc=%.4f  recall=%.4f  f1=%.4f" % (
        name,
        accuracy_score(y_, pred),
        recall_score(y_, pred),
        f1_score(y_, pred),
    ))

print()
print(
    classification_report(
        y_test,
        best_rf.predict(X_test),
        target_names=["ham", "spam"]
    )
)

Train  acc=1.0000  recall=1.0000  f1=1.0000
Val    acc=0.9881  recall=0.9901  f1=0.9881
Test   acc=0.9905  recall=0.9933  f1=0.9907

              precision    recall  f1-score   support

         ham       0.99      0.99      0.99      3455
        spam       0.99      0.99      0.99      3580

    accuracy                           0.99      7035
   macro avg       0.99      0.99      0.99      7035
weighted avg       0.99      0.99      0.99      7035



In [17]:
# Predictions from the final Random Forest
best_rf = grid_rf.best_estimator_
y_pred = best_rf.predict(X_test)

# Add predictions to the test data
test_results = test_df.copy()
test_results["predicted"] = y_pred

# False Positives: Ham → Spam
false_positives = test_results[
    (test_results["label"] == 0) &
    (test_results["predicted"] == 1)
]

# False Negatives: Spam → Ham
false_negatives = test_results[
    (test_results["label"] == 1) &
    (test_results["predicted"] == 0)
]

print("False Positives:", len(false_positives))
print("False Negatives:", len(false_negatives))

False Positives: 43
False Negatives: 24


In [19]:
misclassified = pd.concat([false_positives, false_negatives]).sort_values("label")
display(misclassified[["combined_text", "label", "predicted"]])

,combined_text,label,predicted
184,ie rant new mix online a 2 and half hour mix o...,0,1
291,christmas present hello thank you samuel for y...,0,1
481,patch file.pod added semicolons hello a friend...,0,1
627,ie rant aussie terrace chant you put the all b...,0,1
912,dental nightmares fwd 'some english people hav...,0,1
...,...,...,...
6206,windows live onecare safety scanner run an onl...,1,0
6506,email august deals save up to 40 off denise au...,1,0
6646,spam tenant debt consolidation if this email i...,1,0
6702,canadianpharmacy makes possible to get quality...,1,0
